# 703 — Mechanism-of-Action Aggregation

## Objective

Propagate the frozen LINCS compound target and mechanism-of-action (MoA) annotations into the Notebook 702 exact-structure and perturbational-hypothesis handoffs.

Mechanism summaries are descriptive only. Each **resolved unique exact structure** contributes at most once to a given program × MoA. Unknown structures retain annotation records but never count automatically as independent chemical support.

This notebook does not recompute Notebook 701 significance, infer target–MoA pairwise relationships, select favorable representatives, or perform Phase 9 therapeutic prioritization.


In [1]:
# =============================================================================
# Imports and frozen Notebook 703 constants
# =============================================================================

import json
import platform
from importlib.metadata import version as package_version

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
    validate_artifact_registry,
)
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)
PRIMARY_PERTURBAGENS = 7914
PRIMARY_FAMILY_SIZE = 23742
STRUCTURE_FAMILY_SIZE = 23631

print("Notebook 703 specification: FROZEN")
print("Programs:", PROGRAM_IDS)


Notebook 703 specification: FROZEN
Programs: ('CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03')


In [2]:
# =============================================================================
# Resolve and verify registered frozen inputs before analysis
# =============================================================================

INPUT_IDS = (
    "phase1.110.primary_compound_annotation",
    "phase7.702.perturbagen_structure_map",
    "phase7.702.structure_connectivity_summary",
    "phase7.702.perturbational_hypothesis_catalog",
    "phase7.702.analysis_metadata",
)
registry = load_artifact_registry()
missing_ids = [item for item in INPUT_IDS if item not in registry["artifacts"]]
if missing_ids:
    raise RuntimeError("Missing Notebook 703 inputs: " + ", ".join(missing_ids))

input_paths = {
    item: resolve_artifact_path(registry, item)
    for item in INPUT_IDS
}
input_hashes = {}
for item, path in input_paths.items():
    entry = registry["artifacts"][item]
    if entry["status"] != "frozen":
        raise RuntimeError(f"Input is not frozen: {item}")
    actual_hash = calculate_sha256(path)
    if actual_hash != entry["sha256"]:
        raise RuntimeError(f"Frozen input checksum mismatch: {item}")
    input_hashes[item] = actual_hash

contract_path = (
    Paths.root / "docs" / "contracts" / "phase7"
    / "PHASE7_ANALYSIS_CONTRACT.md"
)
contract_sha256 = calculate_sha256(contract_path)

with input_paths["phase7.702.analysis_metadata"].open(
    "r", encoding="utf-8"
) as handle:
    metadata702 = json.load(handle)
if metadata702["contract"]["sha256"] != contract_sha256:
    raise RuntimeError("Notebook 702 contract fingerprint mismatch.")

print("NOTEBOOK 703 FROZEN INPUT FINGERPRINTS: PASS")


NOTEBOOK 703 FROZEN INPUT FINGERPRINTS: PASS


In [3]:
# =============================================================================
# Load only the registered Notebook 110 / 702 handoffs
# =============================================================================

annotations = pd.read_csv(
    input_paths["phase1.110.primary_compound_annotation"],
    low_memory=False,
)
structure_map = pd.read_csv(
    input_paths["phase7.702.perturbagen_structure_map"],
    low_memory=False,
)
structure_summary = pd.read_csv(
    input_paths["phase7.702.structure_connectivity_summary"],
    low_memory=False,
)
catalog = pd.read_parquet(
    input_paths["phase7.702.perturbational_hypothesis_catalog"]
)

required_annotations = {"pert_id", "target", "moa"}
required_structure = {
    "pert_id", "exact_structure_id", "structure_identity_basis",
}
required_summary = {
    "program_id", "exact_structure_id", "C_RL_median",
    "n_pert_ids", "C_RL_iqr", "direction_concordance",
}
if not required_annotations.issubset(annotations.columns):
    raise RuntimeError("Notebook 110 target/MoA columns not available.")
if not required_structure.issubset(structure_map.columns):
    raise RuntimeError("Notebook 702 structure identity columns missing.")
if not required_summary.issubset(structure_summary.columns):
    raise RuntimeError("Notebook 702 structure summary columns missing.")
if len(structure_map) != PRIMARY_PERTURBAGENS:
    raise RuntimeError("Unexpected exact-structure mapping size.")
if len(structure_summary) != STRUCTURE_FAMILY_SIZE:
    raise RuntimeError("Unexpected structure-summary size.")
if len(catalog) != PRIMARY_FAMILY_SIZE:
    raise RuntimeError("Unexpected primary hypothesis catalog size.")
if structure_map["pert_id"].duplicated().any():
    raise RuntimeError("Non-unique perturbagen identity.")
if structure_summary.duplicated(["program_id", "exact_structure_id"]).any():
    raise RuntimeError("Non-unique program/structure summaries.")
if catalog.duplicated(["program_id", "pert_id"]).any():
    raise RuntimeError("Non-unique primary hypotheses.")
if set(annotations["pert_id"]) != set(structure_map["pert_id"]):
    raise RuntimeError("Compound-annotation universe differs from 702.")
if set(catalog["pert_id"]) != set(structure_map["pert_id"]):
    raise RuntimeError("Primary catalog universe differs from 702.")

print("NOTEBOOK 703 INPUT GATE: PASS")


NOTEBOOK 703 INPUT GATE: PASS


## Annotation relations

LINCS `target` and `moa` fields are treated as **separate descriptive annotation dimensions**. The source does not establish target–MoA pairing, so the notebook does not form a target × MoA Cartesian product. Raw nonempty labels are retained verbatim (apart from surrounding whitespace); no undocumented tokenization is applied to source labels. Missing annotations are explicit, not biological negatives.


In [4]:
# =============================================================================
# Normalize provider labels without inventing target–MoA pairing
# =============================================================================

def normalize_annotation(value):
    """Normalize missing provider values, not biological identities."""
    if pd.isna(value):
        return None
    label = str(value).strip()
    if not label or label.casefold() in {"restricted", "nan", "none"}:
        return None
    return label


def json_sorted(values):
    """Encode a deterministic list of unique non-missing identifiers."""
    return json.dumps(
        sorted({str(v) for v in values if pd.notna(v)}),
        ensure_ascii=False, separators=(",", ":"),
    )


# Each source row is interpreted as up to two independent relations.
relation_frames = []
for kind, source_column in (("TARGET", "target"), ("MOA", "moa")):
    relation = annotations[["pert_id", source_column]].copy()
    relation["annotation_type"] = kind
    relation["annotation_value"] = relation[source_column].map(
        normalize_annotation
    )
    relation["annotation_status"] = np.where(
        relation["annotation_value"].notna(), "ANNOTATED", "MISSING"
    )
    relation_frames.append(
        relation[["pert_id", "annotation_type", "annotation_value",
                  "annotation_status"]]
    )

annotation_relations = (
    pd.concat(relation_frames, ignore_index=True)
    .drop_duplicates()
    .sort_values(["pert_id", "annotation_type", "annotation_status",
                  "annotation_value"], kind="mergesort", na_position="last")
    .reset_index(drop=True)
)
if annotation_relations.duplicated(
    ["pert_id", "annotation_type", "annotation_value", "annotation_status"]
).any():
    raise RuntimeError("Duplicate annotation relation after normalization.")


In [5]:
# =============================================================================
# Propagate structure identity without changing the annotation grain
# =============================================================================

# Keep source-row correspondence as provenance, never as a causal pair.
source_pairs = annotations[['pert_id', 'target', 'moa']].copy()
source_pairs['target'] = source_pairs['target'].map(normalize_annotation)
source_pairs['moa'] = source_pairs['moa'].map(normalize_annotation)
source_pairs = source_pairs.drop_duplicates()

def pair_list_json(block):
    '''Serialize original target/MoA co-annotations deterministically.'''
    # pandas may represent missing labels as NaN after reading the source.
    # Convert them explicitly to JSON null before deterministic ordering.
    pairs = [
        {
            "target": None if pd.isna(t) else str(t),
            "moa": None if pd.isna(m) else str(m),
        }
        for t, m in zip(block["target"], block["moa"])
    ]
    return json.dumps(
        sorted(
            pairs,
            key=lambda d: (d["target"] or "", d["moa"] or ""),
        ),
        ensure_ascii=False,
        separators=(",", ":"),
    )

source_pair_lookup = (
    source_pairs.groupby('pert_id', sort=True)
    .apply(pair_list_json)
    .rename('source_target_moa_pairs_json')
    .reset_index()
)

mechanism_annotation_map = annotation_relations.merge(
    structure_map[
        ["pert_id", "exact_structure_id", "structure_identity_basis"]
    ],
    on="pert_id", how="left", validate="many_to_one",
)
mechanism_annotation_map = mechanism_annotation_map.merge(
    source_pair_lookup, on="pert_id", how="left", validate="many_to_one",
)
if mechanism_annotation_map["source_target_moa_pairs_json"].isna().any():
    raise RuntimeError("Missing original source-row correspondence.")
if mechanism_annotation_map["exact_structure_id"].isna().any():
    raise RuntimeError("Unmapped perturbagen in annotation relations.")

# Only resolved full-InChIKey or canonical-SMILES identities can support
# chemical independence. Unknown, ambiguous, and unresolved stay visible.
RESOLVED_IDENTITY_BASES = {
    "FULL_INCHIKEY",
    "CANONICAL_SMILES_FALLBACK",
}
mechanism_annotation_map["eligible_unique_structure"] = (
    mechanism_annotation_map["structure_identity_basis"].isin(
        RESOLVED_IDENTITY_BASES
    )
)
mechanism_annotation_map = mechanism_annotation_map.sort_values(
    ["annotation_type", "annotation_status", "annotation_value", "pert_id"],
    kind="mergesort", na_position="last",
).reset_index(drop=True)

print("ANNOTATION RELATION PROPAGATION: PASS")
print("Deduplicated relations:", f"{len(mechanism_annotation_map):,}")


ANNOTATION RELATION PROPAGATION: PASS
Deduplicated relations: 18,780


## Unique-structure-aware descriptive MoA summaries

Only `ANNOTATED` MoA records with resolved exact chemical identity enter the chemical-support table. Each program × MoA × exact structure is represented once, regardless of how many `pert_id` identifiers supplied the same annotation.

A mechanism with a single resolved structure remains `SINGLETON`; the grouping never creates a new q-value, a significance tier, or a causal claim. Unknown-structure MoA records are retained separately for coverage accounting.


In [6]:
# =============================================================================
# Derive unique MoA/structure membership, never target × MoA joins
# =============================================================================

moa_relations = mechanism_annotation_map.loc[
    mechanism_annotation_map["annotation_type"].eq("MOA")
    & mechanism_annotation_map["annotation_status"].eq("ANNOTATED")
].copy()

eligible_moa = moa_relations.loc[
    moa_relations["eligible_unique_structure"]
].copy()

# Deduplicate before linking to program-level structure summaries.
moa_structure_ids = (
    eligible_moa[["annotation_value", "exact_structure_id"]]
    .drop_duplicates()
    .rename(columns={"annotation_value": "moa"})
)
moa_structure_program = (
    moa_structure_ids.merge(
        structure_summary,
        on="exact_structure_id",
        how="left",
        validate="many_to_many",
    )
)
if moa_structure_program["program_id"].isna().any():
    raise RuntimeError("Resolved MoA structure missing a program summary.")
if moa_structure_program.duplicated(
    ["program_id", "moa", "exact_structure_id"]
).any():
    raise RuntimeError("A structure was counted twice within a MoA/program.")

# Perturbagen membership is tracked independently and is not a count
# of pharmacologically independent evidence.
moa_pert_ids = (
    moa_relations[["annotation_value", "pert_id"]]
    .drop_duplicates()
    .rename(columns={"annotation_value": "moa"})
)


In [7]:
# =============================================================================
# Build full descriptive summaries including sparse and conflicting contexts
# =============================================================================

summary_rows = []
for (program_id, moa), block in moa_structure_program.groupby(
    ["program_id", "moa"], sort=True, dropna=False
):
    values = pd.to_numeric(block["C_RL_median"], errors="coerce")
    observed = values.dropna().to_numpy(dtype=np.float64)
    n_structures = int(block["exact_structure_id"].nunique())
    n_negative = int(np.sum(observed < 0))
    n_positive = int(np.sum(observed > 0))

    # A zero or missing score is not silently assigned a biological direction.
    if len(observed) == 0:
        direction = "NOT_EVALUABLE"
    elif n_negative > 0 and n_positive > 0:
        direction = "DISCORDANT"
    elif n_negative > 0 and n_positive == 0:
        direction = "NEGATIVE_OR_ZERO"
    elif n_positive > 0 and n_negative == 0:
        direction = "POSITIVE_OR_ZERO"
    else:
        direction = "ZERO_ONLY"

    ids = moa_pert_ids.loc[moa_pert_ids["moa"].eq(moa), "pert_id"]
    summary_rows.append({
        "program_id": program_id,
        "moa": moa,
        "n_unique_structures": n_structures,
        "n_pert_ids": int(ids.nunique()),
        "exact_structure_ids_json": json_sorted(block["exact_structure_id"]),
        "pert_ids_json": json_sorted(ids),
        "structure_C_RL_medians_json": json.dumps(
            [
                {
                    "exact_structure_id": str(row["exact_structure_id"]),
                    "C_RL_median": (
                        float(row["C_RL_median"])
                        if pd.notna(row["C_RL_median"]) else None
                    ),
                }
                for _, row in block.sort_values(
                    "exact_structure_id", kind="mergesort"
                ).iterrows()
            ],
            separators=(",", ":"),
        ),
        "C_RL_median_across_structures": (
            float(np.median(observed)) if len(observed) else np.nan
        ),
        "C_RL_min_across_structures": (
            float(np.min(observed)) if len(observed) else np.nan
        ),
        "C_RL_max_across_structures": (
            float(np.max(observed)) if len(observed) else np.nan
        ),
        "C_RL_iqr_across_structures": (
            float(np.subtract(*np.percentile(observed, [75, 25])))
            if len(observed) > 1 else np.nan
        ),
        "fraction_negative_structures": (
            float(n_negative / len(observed)) if len(observed) else np.nan
        ),
        "direction_concordance": direction,
        "structure_evaluable_count": int(len(observed)),
        "recurrence_context": (
            "SINGLETON" if n_structures == 1 else "MULTI_STRUCTURE_DESCRIPTIVE"
        ),
        "within_structure_heterogeneity_json": json.dumps(
            [
                {
                    "exact_structure_id": str(row["exact_structure_id"]),
                    "n_pert_ids": int(row["n_pert_ids"]),
                    "C_RL_iqr": (
                        float(row["C_RL_iqr"])
                        if pd.notna(row["C_RL_iqr"]) else None
                    ),
                    "direction_concordance": str(row["direction_concordance"]),
                }
                for _, row in block.sort_values(
                    "exact_structure_id", kind="mergesort"
                ).iterrows()
            ],
            separators=(",", ":"),
        ),
    })

mechanism_summary = pd.DataFrame(summary_rows)


In [8]:
# =============================================================================
# Include annotation contexts with no resolved chemical support
# =============================================================================

# Retain all observed MoAs even if every source pert_id lacks exact identity.
all_moas = sorted(moa_relations["annotation_value"].dropna().unique())
all_program_moas = pd.MultiIndex.from_product(
    [PROGRAM_IDS, all_moas], names=["program_id", "moa"]
).to_frame(index=False)

if mechanism_summary.empty:
    mechanism_summary = pd.DataFrame(
        columns=[
            "program_id", "moa", "n_unique_structures", "n_pert_ids",
            "exact_structure_ids_json", "pert_ids_json",
            "structure_C_RL_medians_json", "C_RL_median_across_structures",
            "C_RL_min_across_structures", "C_RL_max_across_structures",
            "C_RL_iqr_across_structures", "fraction_negative_structures",
            "direction_concordance", "structure_evaluable_count",
            "recurrence_context", "within_structure_heterogeneity_json",
        ]
    )

mechanism_summary = all_program_moas.merge(
    mechanism_summary, on=["program_id", "moa"], how="left",
    validate="one_to_one",
)
unresolved_moa = moa_relations.loc[
    ~moa_relations["eligible_unique_structure"]
].groupby("annotation_value")["pert_id"].agg(
    lambda x: sorted(set(x.astype(str)))
).to_dict()

for ix, row in mechanism_summary.iterrows():
    moa = row["moa"]
    unresolved = unresolved_moa.get(moa, [])
    mechanism_summary.at[ix, "unresolved_pert_ids_json"] = json.dumps(
        unresolved, separators=(",", ":"),
    )
    mechanism_summary.at[ix, "n_unresolved_pert_ids"] = len(unresolved)

no_chemistry = mechanism_summary["n_unique_structures"].isna()
mechanism_summary.loc[no_chemistry, "n_unique_structures"] = 0
mechanism_summary.loc[no_chemistry, "n_pert_ids"] = 0
mechanism_summary.loc[no_chemistry, "structure_evaluable_count"] = 0
mechanism_summary.loc[no_chemistry, "recurrence_context"] = "UNRESOLVED_ONLY"
mechanism_summary.loc[no_chemistry, "direction_concordance"] = "NOT_EVALUABLE"
for field in (
    "exact_structure_ids_json", "pert_ids_json",
    "structure_C_RL_medians_json", "within_structure_heterogeneity_json",
):
    mechanism_summary[field] = mechanism_summary[field].fillna("[]")

for col in (
    "n_unique_structures", "n_pert_ids", "structure_evaluable_count",
    "n_unresolved_pert_ids",
):
    mechanism_summary[col] = mechanism_summary[col].astype(int)

mechanism_summary["program_index"] = mechanism_summary["program_id"].map(
    {pid: i for i, pid in enumerate(PROGRAM_IDS)}
)
mechanism_summary = mechanism_summary.sort_values(
    ["program_index", "moa"], kind="mergesort"
).drop(columns="program_index").reset_index(drop=True)


In [9]:
# =============================================================================
# Validate biological and statistical interpretation boundaries
# =============================================================================

if mechanism_summary.duplicated(["program_id", "moa"]).any():
    raise RuntimeError("Duplicate mechanism summary keys.")
if moa_structure_program.duplicated(
    ["program_id", "moa", "exact_structure_id"]
).any():
    raise RuntimeError("Mechanism summary counts duplicate structures.")
if (
    mechanism_summary["n_unique_structures"] < 0
).any():
    raise RuntimeError("Invalid unique-structure counts.")

for _, row in mechanism_summary.iterrows():
    if row["n_unique_structures"] != len(
        json.loads(row["exact_structure_ids_json"])
    ):
        raise RuntimeError("Unique-structure count does not match membership.")
    if row["n_pert_ids"] != len(json.loads(row["pert_ids_json"])):
        raise RuntimeError("Perturbagen count does not match membership.")
if any(
    col.startswith(("p_", "q_")) or "significance" in col.lower()
    for col in mechanism_summary.columns
):
    raise RuntimeError("Forbidden mechanism-level inference field.")
if set(mechanism_annotation_map["annotation_type"]) != {"MOA", "TARGET"}:
    raise RuntimeError("One annotation dimension is missing.")

print("NOTEBOOK 703 MECHANISM AGGREGATION: VALIDATED")
display(
    mechanism_summary["recurrence_context"]
    .value_counts(dropna=False)
    .rename_axis("recurrence_context")
    .reset_index(name="n_program_moa")
)


NOTEBOOK 703 MECHANISM AGGREGATION: VALIDATED


,recurrence_context,n_program_moa
0,MULTI_STRUCTURE_DESCRIPTIVE,1095
1,SINGLETON,720


## Persistence, provenance, and registry publication

Only the three contracted outputs are persisted. During development of Notebook 703, its three artifacts may be regenerated with refreshed registry fingerprints. This does not modify frozen upstream inputs. Metadata describes annotation granularity, unresolved identities, and the prohibition against mechanism-level statistical rescue.


In [10]:
# =============================================================================
# Prepare stable output paths and frozen-identity write guard
# =============================================================================

output_dir = Paths.perturbational_hypotheses
output_dir.mkdir(parents=True, exist_ok=True)
output_paths = {
    "phase7.703.mechanism_annotation_map": (
        output_dir / "703_mechanism_annotation_map.csv"
    ),
    "phase7.703.mechanism_summary": (
        output_dir / "703_mechanism_summary.parquet"
    ),
    "phase7.703.analysis_metadata": (
        output_dir / "703_analysis_metadata.json"
    ),
}
REPUBLISH_703_DURING_DEVELOPMENT = True
frozen_entries = load_artifact_registry()["artifacts"]


def reuse_if_frozen(artifact_id):
    """Reject missing or modified published files before any write."""
    if REPUBLISH_703_DURING_DEVELOPMENT:
        return False
    entry = frozen_entries.get(artifact_id)
    if entry is None:
        return False
    path = output_paths[artifact_id]
    if entry["status"] != "frozen":
        raise RuntimeError(f"Registered artifact is not frozen: {artifact_id}")
    if entry["path"] != project_relative_path(path):
        raise RuntimeError(f"Frozen path mismatch: {artifact_id}")
    if not path.is_file() or calculate_sha256(path) != entry["sha256"]:
        raise RuntimeError(f"Frozen output identity mismatch: {artifact_id}")
    return True


reuse_flags = {item: reuse_if_frozen(item) for item in output_paths}


In [11]:
# =============================================================================
# Prepare deterministic analysis metadata prior to persistence
# =============================================================================

software = {
    "python": platform.python_version(),
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "pyarrow": package_version("pyarrow"),
}
analysis_metadata = {
    "schema_version": 1,
    "notebook": "703_mechanism_of_action_aggregation",
    "contract": {
        "path": project_relative_path(contract_path),
        "sha256": contract_sha256,
    },
    "input_artifact_hashes": input_hashes,
    "program_order": list(PROGRAM_IDS),
    "source_annotation_columns": ["target", "moa"],
    "annotation_relation_grain": (
        "pert_id x annotation_type x source annotation label"
    ),
    "target_moa_cartesian_pairing": False,
    "source_row_target_moa_correspondence_preserved": True,
    "source_row_pairing_role": "provider provenance only; not causal",
    "annotation_missingness_interpretation": "unknown, not biological absence",
    "annotation_tokenization": (
        "none; source labels preserved apart from whitespace trimming"
    ),
    "mechanism_summary_grain": "program_id x source MoA label",
    "chemical_support_unit": "resolved exact_structure_id",
    "unique_structure_counted_once_per_program_moa": True,
    "unresolved_chemistry_role": (
        "retained as perturbagen-level annotation context, not "
        "independent chemical support"
    ),
    "recurrence_threshold_for_strength": None,
    "mechanism_level_inference": False,
    "primary_inference_modified": False,
    "phase5_phase6_used": False,
    "software": software,
    "stable_outputs": list(output_paths),
    "interpretation_limitations": [
        "Provider target and MoA annotations are incomplete and descriptive.",
        "No pairing between a target label and a MoA label is inferred.",
        "Mechanism recurrence does not imply independent pharmacology.",
        "Singleton, discordant, sparse, unknown and heterogeneous "
        "contexts are retained without rescue.",
        "No therapeutic efficacy, causal target or clinical prediction claim.",
    ],
}


In [12]:
# =============================================================================
# Persist or reuse immutable Notebook 703 handoffs
# =============================================================================

if not reuse_flags["phase7.703.mechanism_annotation_map"]:
    mechanism_annotation_map.to_csv(
        output_paths["phase7.703.mechanism_annotation_map"],
        index=False,
    )
if not reuse_flags["phase7.703.mechanism_summary"]:
    mechanism_summary.to_parquet(
        output_paths["phase7.703.mechanism_summary"],
        index=False,
    )
if not reuse_flags["phase7.703.analysis_metadata"]:
    with output_paths["phase7.703.analysis_metadata"].open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(analysis_metadata, handle, indent=2, sort_keys=True)
        handle.write("\n")

print("NOTEBOOK 703 STABLE HANDOFFS: PERSISTED OR REUSED")


NOTEBOOK 703 STABLE HANDOFFS: PERSISTED OR REUSED


In [13]:
# =============================================================================
# Re-read stored outputs and fingerprint exact persisted bytes
# =============================================================================

read_map = pd.read_csv(
    output_paths["phase7.703.mechanism_annotation_map"], low_memory=False
)
read_summary = pd.read_parquet(
    output_paths["phase7.703.mechanism_summary"]
)
with output_paths["phase7.703.analysis_metadata"].open(
    "r", encoding="utf-8"
) as handle:
    read_metadata = json.load(handle)

if read_map.shape != mechanism_annotation_map.shape:
    raise RuntimeError("Persisted annotation-map shape mismatch.")
if read_summary.shape != mechanism_summary.shape:
    raise RuntimeError("Persisted mechanism-summary shape mismatch.")
if read_map.columns.tolist() != mechanism_annotation_map.columns.tolist():
    raise RuntimeError("Persisted annotation-map columns differ.")
if read_summary.columns.tolist() != mechanism_summary.columns.tolist():
    raise RuntimeError("Persisted mechanism-summary columns differ.")
if read_metadata != analysis_metadata:
    raise RuntimeError("Persisted Notebook 703 metadata mismatch.")

output_shapes = {
    "phase7.703.mechanism_annotation_map": list(read_map.shape),
    "phase7.703.mechanism_summary": list(read_summary.shape),
    "phase7.703.analysis_metadata": None,
}
output_hashes = {
    item: calculate_sha256(p) for item, p in output_paths.items()
}
output_sizes = {
    item: int(p.stat().st_size) for item, p in output_paths.items()
}
print("NOTEBOOK 703 OUTPUT FINGERPRINTING: PASS")


NOTEBOOK 703 OUTPUT FINGERPRINTING: PASS


In [14]:
# =============================================================================
# Register exactly three stable handoffs with full lineage
# =============================================================================

NOTEBOOK_703_PATH = (
    "notebooks/phase7_perturbational_hypotheses/"
    "703_mechanism_of_action_aggregation.ipynb"
)
dependency_ids = {
    "phase7.703.mechanism_annotation_map": [
        "phase1.110.primary_compound_annotation",
        "phase7.702.perturbagen_structure_map",
    ],
    "phase7.703.mechanism_summary": [
        "phase1.110.primary_compound_annotation",
        "phase7.702.perturbagen_structure_map",
        "phase7.702.structure_connectivity_summary",
        "phase7.702.perturbational_hypothesis_catalog",
    ],
    "phase7.703.analysis_metadata": list(INPUT_IDS),
}
updated_registry = load_artifact_registry()
for item, path in output_paths.items():
    entry = {
        "path": project_relative_path(path),
        "phase": 7,
        "status": "frozen",
        "artifact_role": "metadata" if item.endswith(
            ".analysis_metadata"
        ) else "handoff",
        "producer": {"type": "notebook", "path": NOTEBOOK_703_PATH},
        "shape": output_shapes[item],
        "size_bytes": output_sizes[item],
        "sha256": output_hashes[item],
        "inputs": [
            {"type": "artifact", "artifact_id": source}
            for source in dependency_ids[item]
        ],
    }
    prior = updated_registry["artifacts"].get(item)
    if prior is not None and prior != entry and not REPUBLISH_703_DURING_DEVELOPMENT:
        raise RuntimeError(f"Non-identical frozen registry entry: {item}")
    updated_registry["artifacts"][item] = entry

validate_artifact_registry(updated_registry)
Paths.artifact_registry.write_text(
    json.dumps(updated_registry, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("NOTEBOOK 703 ARTIFACT REGISTRATION: PASS")


NOTEBOOK 703 ARTIFACT REGISTRATION: PASS


In [15]:
# =============================================================================
# Final Notebook 703 QA: registry, coverage and analytical boundaries
# =============================================================================

final_registry = load_artifact_registry()
qa = {
    "three_registered_outputs": all(
        item in final_registry["artifacts"] for item in output_paths
    ),
    "input_fingerprints_verified": len(input_hashes) == len(INPUT_IDS),
    "annotation_keys_unique": not mechanism_annotation_map.duplicated(
        ["pert_id", "annotation_type", "annotation_value",
         "annotation_status"]
    ).any(),
    "all_perturbagens_preserved": (
        set(mechanism_annotation_map["pert_id"])
        == set(structure_map["pert_id"])
    ),
    "mechanism_keys_unique": not mechanism_summary.duplicated(
        ["program_id", "moa"]
    ).any(),
    "resolved_structures_counted_once": not moa_structure_program.duplicated(
        ["program_id", "moa", "exact_structure_id"]
    ).any(),
    "unknown_structure_not_independent": not (
        eligible_moa["structure_identity_basis"].isin(
            {"UNKNOWN_PERT_ID", "UNRESOLVED_PERT_ID"}
        )
    ).any(),
    "primary_catalog_unchanged": len(catalog) == PRIMARY_FAMILY_SIZE,
    "source_row_pairs_preserved": mechanism_annotation_map[
        "source_target_moa_pairs_json"
    ].notna().all(),
    "no_target_moa_cartesian_pairs": (
        "target" not in mechanism_summary.columns
    ),
    "no_mechanism_pq": not any(
        col.startswith(("p_", "q_")) for col in mechanism_summary.columns
    ),
}
for item, path in output_paths.items():
    entry = final_registry["artifacts"].get(item, {})
    qa[item + ":sha256"] = (
        entry.get("sha256") == calculate_sha256(path)
    )
    qa[item + ":path"] = (
        entry.get("path") == project_relative_path(path)
    )

validate_artifact_registry(final_registry)
qa_series = pd.Series(qa, name="PASS")
display(qa_series)
if not qa_series.all():
    raise RuntimeError(
        "Notebook 703 closure QA failed: "
        + ", ".join(qa_series.index[~qa_series])
    )
print("NOTEBOOK 703 FINAL QA: PASS")
print("Annotated perturbagens:", mechanism_annotation_map["pert_id"].nunique())
print("Program × MoA summaries:", len(mechanism_summary))
print("Registered outputs: 3/3")


three_registered_outputs                      True
input_fingerprints_verified                   True
annotation_keys_unique                        True
all_perturbagens_preserved                    True
mechanism_keys_unique                         True
resolved_structures_counted_once              True
unknown_structure_not_independent             True
primary_catalog_unchanged                     True
source_row_pairs_preserved                    True
no_target_moa_cartesian_pairs                 True
no_mechanism_pq                               True
phase7.703.mechanism_annotation_map:sha256    True
phase7.703.mechanism_annotation_map:path      True
phase7.703.mechanism_summary:sha256           True
phase7.703.mechanism_summary:path             True
phase7.703.analysis_metadata:sha256           True
phase7.703.analysis_metadata:path             True
Name: PASS, dtype: bool

NOTEBOOK 703 FINAL QA: PASS
Annotated perturbagens: 7914
Program × MoA summaries: 1815
Registered outputs: 3/3


## Notebook completion

Notebook 703 is complete only after the final QA cell passes on the frozen local handoffs. Descriptive MoA summaries preserve singleton and discordant contexts and do not redefine Notebook 701 inference. The biological cross-evidence synthesis and therapeutic prioritization remain assigned to Phase 9.
